In [ ]:
import os
import numpy as np

# Setup the module path.
import sys
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt_contrib")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt_analysis")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\mirfusion_library")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\mirxics_jax")

import xicsrt
from xicsrt import xicsrt_config, xicsrt_io
from xicsrt.tools import xicsrt_spline

from w7x_npablant import xicsrt_w7x_lalston

In [ ]:
def build_training_set_configs(num_training_set=10, base_seed=1234, output_path=None):
    """
    Generate randomized plasma profile configurations and save one configuration file for each training sample.

    Parameters
    ----------
    num_training_set : number of randomized configuration files to generate.
    base_seed : starting random seed used to generate reproducible plasma profile configurations.
    output_path : directory in which the configuration files are saved.
    """

    if output_path is None:
        raise ValueError("There is no output_path listed. Config files failed to save.")
        return

    os.makedirs(output_path, exist_ok=True)

    for ii in range(num_training_set):

        # Give every sample its own reproducible seed.
        profile_seed = base_seed + ii
        np.random.seed(profile_seed)

        # Create the base ray-tracing configuration.
        config = xicsrt_w7x_lalston.get_config()
        config = xicsrt_w7x_lalston.initialize(config)

        # Generate the randomized spline dictionaries.
        generated_profiles = {
            "profile_ion_temp": xicsrt_spline.generate_random_ion_temp(),
            "profile_electron_temp": xicsrt_spline.generate_random_electron_temp(),
            "profile_emissivity": xicsrt_spline.generate_random_emissivity(),
            "profile_perpendicular_velocity": xicsrt_spline.generate_random_perpendicular_velocity(),
            "profile_parallel_velocity": xicsrt_spline.generate_random_parallel_velocity(),
        }

        # Insert the generated profiles into the configuration and update it.
        config_new = {
            "sources": {
                "plasma": generated_profiles,
            }
        }
        xicsrt_config.update_config(config, config_new, strict=False, update=True,)

        # Store useful identifying information.
        config.setdefault("scenario", {})
        config["scenario"]["sample_index"] = ii
        config["scenario"]["profile_seed"] = int(profile_seed)

        filename = f"raytrace_config_{ii:06d}.json"

        xicsrt.xicsrt_io.save_config(
            config,
            filename=filename,
            path=output_path,
            mkdir=True,
            overwrite=False,
        )

        if (ii + 1) % 100 == 0:
            print(
                f"Saved {ii + 1:,} of "
                f"{num_training_set:,} configurations."
            )

In [ ]:
output_path = r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xics_ml_pipeline\nn_training_data\xicsrt_profile_configs"
build_training_set_configs(num_training_set=10, base_seed=1234, output_path=output_path)